# ARLoop-1.2B — looped transformer text completion on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/arloop-1.2b-looped-transformer-1f78/colab/ARLoop-1.2B-looped-transformer.ipynb)

[`osunlp/ARLoop-1.2B`](https://huggingface.co/osunlp/ARLoop-1.2B) is OSU NLP Group's **fully-open pretrained looped transformer with attention residuals** (Apache-2.0, public, **1.2B** params, `custom_code`). 24 physical layers are shared across **3** recurrent passes (**effective depth 72**), with a **4096** context window and the **SmolLM2** tokenizer (vocab 49,152). The Hub release is the final checkpoint of a **500B-token** training run.

It is a **base** pretrained model for **text completion**, not a chat-tuned assistant. There is no chat template. The card says it can be used for mid-training and post-training. A smaller sibling [`osunlp/ARLoop-0.6B`](https://huggingface.co/osunlp/ARLoop-0.6B) (18 physical layers × 3 recurrences, 180B tokens) exists; this notebook stays on 1.2B. Intermediate checkpoints are Hub revisions such as `step-010000`.

**Why this model.** Looped transformers reuse one stack of layers several times so effective depth grows without storing that many unique layers. ARLoop adds **attention residuals**: each layer's input is a learned mix of recent residual blocks rather than a single summed stream. The portable Hub file (`modeling_arloop.py`) runs on PyTorch SDPA and says no training repo, FLA, Triton, or flash-attn install is needed.

The paper ([arXiv:2611.05234](https://arxiv.org/abs/2611.05234)) reports that this 1.2B model, pretrained on 0.5T tokens with three recurrences, outperforms Huginn-3.5B across the authors' evaluated tasks and is competitive with similar-sized non-looped models such as Qwen3.5-2B despite less training data, with particularly strong improvements on math and coding. Those are **paper numbers**, not measurements from this notebook.

**Use cases:** [when to pick this notebook](https://github.com/donvito/notebooks/blob/main/docs/ARLoop-1.2B-looped-transformer.md)

**Licence:** Apache-2.0. Not gated.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about **15 GB** of VRAM and **no native bfloat16**. The card loads `dtype=torch.bfloat16`. This notebook loads **float16** (~2.4 GB of parameters). If logits are NaNs or the text is garbage, it reloads **float32** (~5 GB of parameters), which still fits. Stored shards are about **4.9 GB** of original FP32 weights (that is the download). The custom forward still autocasts activations to bfloat16 (emulated on a T4) and keeps routing / normalization statistics in FP32.

`trust_remote_code=True` is required. The 1.2B card pins **`transformers==5.12.1`**. After the install cell, **restart the session** so Colab's already-imported transformers is replaced. Do **not** reinstall torch.

**Links:** [model card](https://huggingface.co/osunlp/ARLoop-1.2B) · [0.6B sibling](https://huggingface.co/osunlp/ARLoop-0.6B) · [paper](https://arxiv.org/abs/2611.05234) · [code](https://github.com/OSU-NLP-Group/ARLoop) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/ARLoop-1.2B-looped-transformer.md) · [source post](https://x.com/yuekun_yao/status/2119982422259622086)


## 1. Install the card pin (keep Colab's torch)

The 1.2B card is:

```bash
pip install 'transformers==5.12.1' safetensors accelerate
# Install the PyTorch build appropriate for your CUDA environment separately.
```

Colab already has a CUDA torch. **Do not `pip install torch`.** Use `%pip`, not `python -I -m pip` (isolated mode drops Colab site-packages). Do **not** install `flash-attn`, `flash-linear-attention`, or Triton — the Hub inference file is portable SDPA.

This pin may **downgrade** Colab's preinstalled transformers. That is intentional (the checkpoint was saved with 5.12.1). After it finishes you **must** restart the runtime.


In [ ]:
%pip install -q "transformers==5.12.1" safetensors accelerate

# Card: pip install 'transformers==5.12.1' safetensors accelerate
# Do NOT pip install torch. Colab already ships a CUDA wheel; forcing a
# reinstall often drops GPU support or pulls a CPU-only build.
# Do NOT install flash-attn / FLA / Triton (T4 has no FA2; Hub code is SDPA).
# Use %pip, not python -I -m pip (isolated mode drops Colab site-packages).
print("install done — restart the runtime before the GPU-check cell")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to **5.12.1** is invisible until the process restarts. You do **not** need to rerun the install cell after the restart unless it failed.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. A free T4 (~15 GB, `sm_75`) is the path this notebook is written for. The next cell stops if you are on CPU.

T4 has **no native bfloat16**. Weights load in **float16**. Set `DTYPE = torch.float32` before the load cell if you already know float16 was garbage on this runtime.


In [ ]:
import os
import gc
import time

import torch
import transformers

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("torch", torch.__version__)

loaded = transformers.__version__.split("+")[0]
assert loaded == "5.12.1", (
    f"transformers {transformers.__version__} is loaded; the ARLoop-1.2B card pins "
    "transformers==5.12.1. Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Hub code uses PyTorch SDPA.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
MODEL_ID = "osunlp/ARLoop-1.2B"
# Card default is torch.bfloat16. T4 has no native bf16 tensor cores.
DTYPE = torch.float16
major, minor = torch.cuda.get_device_capability()
free_b, total_b = torch.cuda.mem_get_info()
props = torch.cuda.get_device_properties(0)

print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (card default is bfloat16; T4 has no native bf16)")
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print(f"device capability {major}.{minor} | multiprocessors {props.multi_processor_count}")
if major < 8:
    print("No native bfloat16 on this GPU — loading float16 weights.")
else:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")
print(
    "Note: modeling_arloop.py still autocasts activations to bfloat16 "
    "(emulated on T4) and keeps routing / RMSNorm stats in FP32."
)


def print_memory(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


## 3. Load ARLoop-1.2B

`AutoTokenizer` + `AutoModelForCausalLM.from_pretrained(..., trust_remote_code=True, dtype=...)` as on the card, then `.to("cuda").eval()`. First run downloads about **4.9 GB** (three FP32 safetensor shards). Float16 weights in VRAM should be about **2.4 GB**; float32 fallback about **5 GB**.

After load, a short forward pass checks that logits are finite. If they are not, the cell reloads in float32.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_arloop(revision=None, dtype=None):
    dtype = DTYPE if dtype is None else dtype
    kwargs = dict(trust_remote_code=True, dtype=dtype)
    if revision is not None:
        kwargs["revision"] = revision
    label = revision or "main"
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).to(DEVICE).eval()
    torch.cuda.synchronize()
    print(f"loaded {MODEL_ID} revision={label} dtype={dtype} in {time.perf_counter() - t0:.1f}s")
    print("architecture:", getattr(mdl.config, "architectures", None))
    print(
        "physical_layers:", mdl.config.physical_layers,
        "| loop_count:", mdl.config.loop_count,
        "| effective depth (num_hidden_layers):", mdl.config.num_hidden_layers,
    )
    print(
        "hidden_size:", mdl.config.hidden_size,
        "| context:", mdl.config.max_position_embeddings,
        "| vocab:", mdl.config.vocab_size,
    )
    n_params = sum(p.numel() for p in mdl.parameters()) / 1e9
    print(f"parameters: {n_params:.3f}B | param dtype: {next(mdl.parameters()).dtype} | device: {mdl.device}")
    print_memory("after load  ")
    return mdl


def logits_are_finite(mdl, prompt="The capital of France is", tok=None):
    tok = tokenizer if tok is None else tok
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        logits = mdl(**inputs).logits
    ok = bool(torch.isfinite(logits).all().item())
    print(
        f"finite logits: {ok} | shape {tuple(logits.shape)} | "
        f"min {logits.min().item():.3g} | max {logits.max().item():.3g}"
    )
    return ok


model = load_arloop()
if not logits_are_finite(model):
    print("Non-finite logits in float16 — reloading in float32 (~5 GB, still T4-sized).")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    DTYPE = torch.float32
    model = load_arloop(dtype=DTYPE)
    if not logits_are_finite(model):
        raise RuntimeError(
            "Logits are still non-finite in float32. The custom forward autocasts "
            "activations to bfloat16 even when weights are float32; inspect the output "
            "or try a GPU with native bf16."
        )

print("Ready. If later completions look like garbage, set DTYPE = torch.float32 and rerun this cell.")


## 4. Demo 1 — factual completion

The card's own prompt is `The capital of France is`, greedy, `max_new_tokens=64`. A second short factual stem is included. This is **completion**, not chat: no system prompt, no chat template.


In [ ]:
def complete(prompt, mdl=None, tok=None, max_new_tokens=64):
    mdl = model if mdl is None else mdl
    tok = tokenizer if tok is None else tok
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.inference_mode():
        output = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize()
    text = tok.decode(output[0], skip_special_tokens=True)
    print(f"{time.perf_counter() - t0:.2f}s | new tokens: {output.shape[-1] - inputs['input_ids'].shape[-1]}")
    print_memory("after generate  ")
    return text


FACTUAL_PROMPTS = [
    "The capital of France is",
    "The chemical symbol for gold is",
]

final_completions = {}
for prompt in FACTUAL_PROMPTS:
    print("=" * 72)
    print("PROMPT:", prompt)
    print("COMPLETION:")
    text = complete(prompt)
    final_completions[prompt] = text
    print(text)
    print()


## 5. Demo 2 — few-shot pattern completion

A base model continues the pattern in the prompt. Keep the stem short; greedy decode, 32 new tokens.


In [ ]:
PATTERN_PROMPTS = [
    "France → Paris\nGermany → Berlin\nItaly → Rome\nSpain →",
    "2, 4, 8, 16, 32,",
]

for prompt in PATTERN_PROMPTS:
    print("=" * 72)
    print("PROMPT:")
    print(prompt)
    print("COMPLETION:")
    print(complete(prompt, max_new_tokens=32))
    print()


## 6. Demo 3 — early training checkpoint (`step-010000`)

`main` is the final 1.2B checkpoint (~500B tokens, optimizer step 238,417). Revision [`step-010000`](https://huggingface.co/osunlp/ARLoop-1.2B/tree/step-010000) is 10,000 steps / about **21B** tokens. The card says each revision is independently loadable; use the same revision for the model and tokenizer.

This cell **releases** the final model first so a free T4 is not holding two 1.2B copies. Same card prompt, same greedy settings. This is a training-progress smoke test, not a benchmark.

The 0.6B sibling is not loaded here: its card pins `transformers==5.22.0`, and this runtime is pinned to the 1.2B card's 5.12.1.


In [ ]:
EARLY_REVISION = "step-010000"
CARD_PROMPT = "The capital of France is"

print("Releasing the final checkpoint before loading", EARLY_REVISION)
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_memory("after release  ")

early_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, revision=EARLY_REVISION
)
if early_tokenizer.pad_token_id is None:
    early_tokenizer.pad_token = early_tokenizer.eos_token

early = load_arloop(revision=EARLY_REVISION, dtype=DTYPE)
if not logits_are_finite(early, CARD_PROMPT, tok=early_tokenizer):
    print("Non-finite logits on the early checkpoint — retrying float32.")
    del early
    gc.collect()
    torch.cuda.empty_cache()
    early = load_arloop(revision=EARLY_REVISION, dtype=torch.float32)

early_text = complete(CARD_PROMPT, mdl=early, tok=early_tokenizer)
print("=" * 72)
print("PROMPT:", CARD_PROMPT)
print()
print("--- main (final, ~500B tokens) ---")
print(final_completions.get(CARD_PROMPT, "(re-run Demo 1 to fill this)"))
print()
print(f"--- {EARLY_REVISION} (~21B tokens) ---")
print(early_text)
print()
print(
    "Early-step text is often less fluent; that is expected "
    "(~21B vs ~500B tokens), not a notebook bug."
)
model = None  # final weights were released; early checkpoint is in `early`


## 7. Cleanup

Release the checkpoint so later cells (or a second model) get the VRAM back.


In [ ]:
print_memory("before cleanup  ")
for name in ("model", "early", "tokenizer", "early_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_memory("after cleanup   ")
print("Released. Re-run the load cell if you want another completion.")


## Caveats

**Base model, not chat.** No instruction tuning, no chat template, no tools. Prompt it as a completer. For a chat-tuned T4 LM see [LFM3-1.7B](LFM3-1.7B-chat-tools.ipynb).

**`trust_remote_code`.** [`modeling_arloop.py`](https://huggingface.co/osunlp/ARLoop-1.2B/blob/main/modeling_arloop.py) and [`configuration_arloop.py`](https://huggingface.co/osunlp/ARLoop-1.2B/blob/main/configuration_arloop.py) run locally. Review them, and pin a revision for reproduction.

**Transformers pin.** `transformers==5.12.1` (1.2B card). Restart the session after the install cell. Do not reinstall Colab's torch. The 0.6B sibling card pins `transformers==5.22.0` — do not mix those pins in one runtime.

**T4 / dtype.** Card loads bfloat16. This notebook loads float16, with a float32 fallback (~5 GB) if logits are non-finite. Custom code still autocasts activations to bfloat16 (emulated on `sm_75`) and keeps attention-residual routing in FP32. Paper / card quality numbers used bf16 on datacenter GPUs; this T4 path is not those numbers.

**VRAM assumption.** Download ~4.9 GB (FP32 shards). Float16 parameters ~2.4 GB; float32 ~5 GB. KV cache is indexed by **effective** layer (72), so keep `max_new_tokens` small. Estimates, not a measured Colab log attached to this file.

**Checkpoints.** `main` ≈ 500B tokens. `step-010000` ≈ 21B tokens. The card lists revisions through `step-230000` plus `step-238417` / `main`.

**Related looped notebook.** [`oruk-loop-153M-looped-asr.ipynb`](oruk-loop-153M-looped-asr.ipynb) is a looped **ASR** student, not a text LM, and its weights are not Apache-2.0.
